Here chi=20 - no damping, fixing the number of Gilt R-matrix iterations. Newton method converges, starting from T=T_c, doing 4 RG iterations, then Newton. Using 5 eigenvalues here

In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5 # from the paper for this chi
chi = 20
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 4 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 4, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[5], accepted_elements, _ = fix_discrete_gauge(traj[5]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array 

In [ ]:
Newton iterations (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [3]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 5, accepted_elements[i], gilt_pars);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

i=1
||R(A[i])-A[i]||= 0.05785177411987096
Dict{Any, Any}((1, "N") => 28, (1, "W") => 22, (1, "S") => 26, (1, "E") => 22, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  7 eigenvalues converged
│ *  norm of residuals = (1.9888130091058864e-40, 5.707343325550389e-26, 2.676745536127191e-25, 3.0092883113935614e-17, 5.141485108901361e-16, 6.425741184727582e-14, 6.425741184727582e-14)
└ *  number of operations = 44


EIGENVALUES (INITIAL):
1.9863026246747197 + 0.0im
-0.9355985296498324 + 0.0im
-0.9247487485530992 + 0.0im
0.5554839426161478 + 0.0im
0.5499380293331942 + 0.0im
0.0019598759147831176 + 0.4022132076005157im
0.0019598759147831176 - 0.4022132076005157im
||deltaA[i]||= 0.12867019012928455
i=2
||R(A[i])-A[i]||= 0.026989129333975497
Dict{Any, Any}((1, "N") => 68, (1, "W") => 44, (1, "S") => 50, (1, "E") => 44, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  7 eigenvalues converged
│ *  norm of residuals = (6.978967955986465e-37, 1.2376297076464252e-23, 1.8480749464023446e-22, 6.9066253366365344e-18, 4.372160160279965e-16, 1.7932640304892574e-15, 1.7932640304892574e-15)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.9887447501268147 + 0.0im
-0.9726844288596342 + 0.0im
-0.9618238019421339 + 0.0im
0.6239292857601034 + 0.0im
0.5687484044047006 + 0.0im
-0.004153208880228312 + 0.5042406009983933im
-0.004153208880228312 - 0.5042406009983933im
||deltaA[i]||= 0.06330906059258061
i=3
||R(A[i])-A[i]||= 0.015803971102111725
Dict{Any, Any}((1, "N") => 86, (1, "W") => 69, (1, "S") => 54, (1, "E") => 67, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


...

┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  7 eigenvalues converged
│ *  norm of residuals = (2.06957983459899e-35, 5.164788172435753e-22, 1.2771068063669216e-22, 1.273204755490205e-16, 5.266382486824483e-16, 1.170586893761736e-14, 1.170586893761736e-14)
└ *  number of operations = 42


EIGENVALUES (INITIAL):
1.9982872865695736 + 0.0im
-0.9933175143405173 + 0.0im
-0.9862643789381624 + 0.0im
0.6302145529646236 + 0.0im
0.5945367471313405 + 0.0im
-0.005888815353862729 + 0.530700545345468im
-0.005888815353862729 - 0.530700545345468im
||deltaA[i]||= 0.0034789249281126336
i=6
||R(A[i])-A[i]||= 0.0012001786849784087
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 45, (1, "E") => 47, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  7 eigenvalues converged
│ *  norm of residuals = (4.404213296773629e-37, 3.821377783565883e-23, 9.148534070470755e-22, 3.896439521537632e-15, 3.896439521537632e-15, 1.1371040858703406e-14, 1.1371040858703406e-14)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.9985706989387073 + 0.0im
-0.9933403523378872 + 0.0im
-0.9849015148094427 + 0.0im
0.5775888895179181 + 0.004804193606630647im
0.5775888895179181 - 0.004804193606630647im
-0.004465351191533165 + 0.5136119125138996im
-0.004465351191533165 - 0.5136119125138996im
||deltaA[i]||= 0.0023593061168569037
i=7
||R(A[i])-A[i]||= 0.0003478292393065974
Dict{Any, Any}((1, "N") => 72, (1, "W") => 49, (1, "S") => 45, (1, "E") => 49, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


LoadError: InterruptException: